In [1]:
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd()
events_filepath = BASE_DIR / "data/raw/events_data"
output_path = BASE_DIR / "data/clean/events.parquet"

In [3]:
events_data = pd.read_parquet(output_path)

In [4]:
events_data

,event_date,event_timestamp,event_name,platform,language,user_id,user_pseudo_id,tour_id,story_id,lang_id,audio_time_played,audio_time_paused
0,20250718,1752821825892003,click_listen_now,ANDROID,pt-pt,<NA>,cdedf0cb5ee77ff6598c1ccc1988e297,616,<NA>,2,NaN,NaN
1,20250718,1752869346776007,click_purchases_tab,ANDROID,it-it,<NA>,3c63844bb1b9b300c1b2199aaca9fc14,<NA>,<NA>,<NA>,NaN,NaN
2,20250718,1752843552913022,story_listened_20,ANDROID,it-it,<NA>,ad7744bf9d7f13f4464dba7b2059aed4,869,53297,4,NaN,NaN
3,20250718,1752818578211035,story_listened_60,ANDROID,pl-pl,<NA>,7eceffaeaaaa494e051d46c6ef418572,403,12529,2,NaN,NaN
4,20250718,1752824669559024,tour_download_progress,ANDROID,es-es,<NA>,73f18dd1750ad51d6486a1af895471c4,644,<NA>,2,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
13471330,20251003,1759480602471002,story_listened_80,ANDROID,fr-fr,<NA>,fec9afe36ab7454e11c0f763f4afed92,858,52417,6,NaN,NaN
13471331,20251003,1759481420845009,story_listened_20,ANDROID,fr-fr,<NA>,fec9afe36ab7454e11c0f763f4afed92,858,52432,6,NaN,NaN
13471332,20251003,1759479561482002,story_completed,ANDROID,fr-fr,<NA>,fec9afe36ab7454e11c0f763f4afed92,858,52394,6,NaN,NaN
13471333,20251003,1759478811852000,play,ANDROID,fr-fr,<NA>,fec9afe36ab7454e11c0f763f4afed92,858,52380,6,00:07,NaN


### Testing pseudo id and language device setting

In [5]:
# Number of distinct languages per pseudo_user_id
lang_per_pseudo = (
    events_data
    .groupby("user_pseudo_id")["language"]
    .nunique()
    .reset_index(name="distinct_languages")
)

# See distribution
lang_per_pseudo["distinct_languages"].value_counts().sort_index()

distinct_languages
1    36773
2       34
Name: count, dtype: int64

In [6]:
# Pseudo users with more than 1 language
multi_lang_users = lang_per_pseudo[lang_per_pseudo["distinct_languages"] > 1]

multi_lang_users

,user_pseudo_id,distinct_languages
1011,06d1601565dd32257942bb81c1161583,2
1172,08086CD5539F4F579EE4F7D65A0BD989,2
2105,0c9c09f34e655f856f35ef6b3f11ef73,2
6120,2A8BAA82CABE4F3786952261D04A7848,2
6150,2AD9AA3E30124A09B6EDC21F2AF3192A,2
8243,3972E3B411F14729BA74BA2CB9AB7F92,2
10832,4BB2C6F707754557BA21D2260655A666,2
11544,503A217992124FD6895D3AB4FCB50188,2
12777,58E199F20177482786A2C379A3E2D9E8,2
13280,5DB3DFE6904F479B84315CEAE8B25353,2


## How many distinct user id's per pseudo user

In [8]:
# Number of distinct user_id per pseudo_user_id
user_per_pseudo = (
    events_data
    .groupby("user_pseudo_id")["user_id"]
    .nunique()
    .reset_index(name="distinct_user_ids")
)

# Distribution
user_per_pseudo["distinct_user_ids"].value_counts().sort_index()

distinct_user_ids
0    13049
1    17383
2     6325
3       48
4        2
Name: count, dtype: int64

In [10]:
# Pseudo users linked to multiple user_ids
multi_user_cases = user_per_pseudo[user_per_pseudo["distinct_user_ids"] > 1]


In [17]:
multi_user_cases.distinct_user_ids.sort_values()

4        2
23966    2
23963    2
23961    2
23959    2
        ..
28876    3
16220    3
13387    3
20342    4
16125    4
Name: distinct_user_ids, Length: 6375, dtype: int64

## How many pseudo_id's per user?

In [18]:
# Number of distinct pseudo ids per user_id
pseudo_per_user = (
    events_data
    .groupby("user_id")["user_pseudo_id"]
    .nunique()
    .reset_index(name="distinct_pseudo_ids")
)

# Distribution
pseudo_per_user["distinct_pseudo_ids"].value_counts().sort_index()

distinct_pseudo_ids
1       21934
2         182
3          11
4           3
5           2
9           1
7823        1
Name: count, dtype: int64

In [19]:
multi_device_users = pseudo_per_user[pseudo_per_user["distinct_pseudo_ids"] > 1]

multi_device_users

,user_id,distinct_pseudo_ids
0,0,7823
5,9821,9
7,10989,2
34,37419,2
43,39975,2
...,...,...
20980,577737,2
21018,577902,2
21606,580534,2
21973,582342,2


## Replace user id == 0 and run analysis again

In [20]:
events_data["user_id"] = events_data["user_id"].replace(0, pd.NA)

In [21]:
# Number of distinct user_id per pseudo_user_id
user_per_pseudo = (
    events_data
    .groupby("user_pseudo_id")["user_id"]
    .nunique()
    .reset_index(name="distinct_user_ids")
)

# Distribution
user_per_pseudo["distinct_user_ids"].value_counts().sort_index()

distinct_user_ids
0    14817
1    21629
2      351
3        9
4        1
Name: count, dtype: int64

## Is platform tied to pseudo_id==device?

In [22]:
# Number of distinct platforms per pseudo user
platform_per_pseudo = (
    events_data
    .groupby("user_pseudo_id")["platform"]
    .nunique()
    .reset_index(name="distinct_platforms")
)

# Distribution
platform_per_pseudo["distinct_platforms"].value_counts().sort_index()

distinct_platforms
1    36807
Name: count, dtype: int64

Yes it is

## Its safe to say that using pseudo_user_id as distinct users is correct